# Linear Regression on Two Datasets (Gradio + Flask)
Put `cafe_sales.xlsx` and `laptop_prices.xlsx` in a `data/` folder next to this notebook.

In [ ]:
# !pip install pandas numpy scikit-learn openpyxl matplotlib gradio flask   # uncomment on first run

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

DATASETS = {
    "Cafe Cold-Drink Sales": ("data/cafe_sales.xlsx", "Cold_Drink_Sales_PKR"),
    "Laptop Prices": ("data/laptop_prices.xlsx", "Price_USD"),
}

## 1. Load and explore the data

In [ ]:
for name, (path, target) in DATASETS.items():
    df = pd.read_excel(path)
    print(f"=== {name} === shape: {df.shape}")
    display(df.head())
    display(df.describe().round(2))
    print()

## 2. Train and evaluate linear regression

In [ ]:
class RegressionModel:
    def __init__(self, name):
        path, self.target = DATASETS[name]
        self.name = name
        self.df = pd.read_excel(path)
        self.features = [c for c in self.df.columns if c != self.target]
        X, y = self.df[self.features], self.df[self.target]
        X_tr, self.X_te, y_tr, self.y_te = train_test_split(X, y, test_size=0.2, random_state=42)
        self.model = LinearRegression().fit(X_tr, y_tr)
        self.pred_te = self.model.predict(self.X_te)
        self.metrics = {
            "R2": round(r2_score(self.y_te, self.pred_te), 4),
            "MAE": round(mean_absolute_error(self.y_te, self.pred_te), 2),
            "RMSE": round(float(np.sqrt(mean_squared_error(self.y_te, self.pred_te))), 2),
        }
        self.coefs = {f: round(float(c), 4) for f, c in zip(self.features, self.model.coef_)}
        self.intercept = round(float(self.model.intercept_), 4)

    def predict(self, values):
        return float(self.model.predict(pd.DataFrame([values], columns=self.features))[0])

    def feature_ranges(self):
        d = self.df[self.features]
        return {f: (float(d[f].min()), float(d[f].max()), float(d[f].median())) for f in self.features}

models = {n: RegressionModel(n) for n in DATASETS}
for n, m in models.items():
    print(n, m.metrics, "| intercept:", m.intercept, "| coefs:", m.coefs)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (n, m) in zip(axes, models.items()):
    ax.scatter(m.y_te, m.pred_te)
    lo, hi = min(m.y_te.min(), m.pred_te.min()), max(m.y_te.max(), m.pred_te.max())
    ax.plot([lo, hi], [lo, hi], "r--")
    ax.set_title(f"{n} (R2={m.metrics['R2']})"); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
plt.tight_layout(); plt.show()

## 3. Gradio interface (appears inside the notebook)

In [ ]:
import gradio as gr

def build_inputs(dataset):
    m = models[dataset]; rngs = m.feature_ranges(); ups = []
    for i in range(5):
        if i < len(m.features):
            f = m.features[i]; lo, hi, med = rngs[f]
            ups.append(gr.update(visible=True, label=f"{f} (range {lo:g}-{hi:g})", value=med))
        else:
            ups.append(gr.update(visible=False, value=None))
    return ups

def summary(dataset):
    m = models[dataset]
    return (f"**Target:** {m.target}  \n**Features:** {', '.join(m.features)}  \n"
            f"**Metrics:** {m.metrics}  \n**Intercept:** {m.intercept}  \n**Coefficients:** {m.coefs}")

def gradio_predict(dataset, *vals):
    m = models[dataset]; vals = vals[:len(m.features)]
    if any(v is None for v in vals): return "Please fill in all inputs."
    return f"Predicted {m.target}: {m.predict(list(vals)):,.2f}"

with gr.Blocks(title="Linear Regression - Gradio") as demo:
    gr.Markdown("# Linear Regression Predictor (Gradio)")
    ds = gr.Dropdown(list(DATASETS), value=list(DATASETS)[0], label="Choose dataset")
    info_md = gr.Markdown()
    inputs = [gr.Number(visible=False) for _ in range(5)]
    btn = gr.Button("Predict", variant="primary")
    out = gr.Textbox(label="Result")
    ds.change(summary, ds, info_md); ds.change(build_inputs, ds, inputs)
    demo.load(summary, ds, info_md);  demo.load(build_inputs, ds, inputs)
    btn.click(gradio_predict, [ds] + inputs, out)

demo.launch()   # shows inline; use share=True for a public link
# demo.close()  # run this to stop the server when you are done

## 4. Flask interface (runs in a background thread)
After running the cell, open http://127.0.0.1:5000 in your browser.

In [ ]:
from flask import Flask, render_template_string, request
from werkzeug.serving import make_server
import threading

PAGE = '''
<!doctype html><html><head><meta charset="utf-8"><title>Linear Regression - Flask</title>
<style>body{font-family:Arial;max-width:800px;margin:30px auto;padding:0 15px}
.card{border:1px solid #ddd;border-radius:8px;padding:15px;margin:15px 0;background:#fafafa}
label{display:block;margin-top:10px;font-weight:bold} input,select{padding:6px;width:260px}
button{margin-top:15px;padding:8px 18px;background:#2c3e50;color:#fff;border:0;border-radius:5px}
.result{font-size:20px;color:#1a7f37;font-weight:bold}</style></head><body>
<h1>Linear Regression Predictor (Flask)</h1>
<form method="get"><label>Choose dataset</label>
<select name="dataset" onchange="this.form.submit()">
{% for d in datasets %}<option value="{{d}}" {% if d==name %}selected{% endif %}>{{d}}</option>{% endfor %}</select></form>
<div class="card"><b>Target:</b> {{m.target}}<br><b>Features:</b> {{m.features|join(', ')}}<br>
<b>Metrics:</b> {{m.metrics}}<br><b>Intercept:</b> {{m.intercept}}<br><b>Coefficients:</b> {{m.coefs}}</div>
<div class="card"><h3>Make a prediction</h3>
<form method="post"><input type="hidden" name="dataset" value="{{name}}">
{% for f in m.features %}<label>{{f}} (range {{'%g'|format(ranges[f][0])}} - {{'%g'|format(ranges[f][1])}})</label>
<input type="number" step="any" name="{{f}}" value="{{entered.get(f, ranges[f][2])}}" required>{% endfor %}
<br><button type="submit">Predict</button></form>
{% if result %}<p class="result">Predicted {{m.target}}: {{result}}</p>{% endif %}
{% if error %}<p style="color:red">{{error}}</p>{% endif %}</div>
<div class="card"><h3>Dataset preview</h3>{{preview|safe}}</div></body></html>
'''

app = Flask(__name__)

@app.route("/", methods=["GET", "POST"])
def index():
    name = request.values.get("dataset", list(DATASETS)[0]); m = models[name]
    result = error = None; entered = {}
    if request.method == "POST":
        try:
            entered = {f: float(request.form[f]) for f in m.features}
            result = f"{m.predict(list(entered.values())):,.2f}"
        except (ValueError, KeyError):
            error = "Please enter valid numbers in all fields."
    return render_template_string(PAGE, datasets=list(DATASETS), name=name, m=m,
        ranges=m.feature_ranges(), preview=m.df.head(10).to_html(index=False),
        result=result, error=error, entered=entered)

server = make_server("127.0.0.1", 5000, app)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("Flask running at http://127.0.0.1:5000")
# server.shutdown()   # run this to stop Flask